# 谢尔宾斯基地毯：主分析笔记本

## 目标
验证维数D的编译指纹，特别是连分数中的巨大偏商。

## 准备工作
确保在项目根目录（Sierpinski_Experiment/）中运行此笔记本。

In [ ]:
# 第一步：设置Python路径
import sys
import os

# 添加项目根目录到Python路径
project_root = os.getcwd()  # 当前目录应该是项目根目录
if not os.path.exists('src'):
    print("❌ 错误：请在项目根目录（Sierpinski_Experiment/）中运行此笔记本")
    print(f"当前目录：{project_root}")
    print("请运行：cd /path/to/Sierpinski_Experiment")
else:
    sys.path.append(project_root)
    print(f"✅ Python路径已设置，项目根目录：{project_root}")

In [ ]:
# 第二步：导入模块
try:
    from src.constants import compute_sierpinski_constants
    from src.continued_fraction import analyze_dimension_cf
    import mpmath as mp
    
    print("✅ 模块导入成功！")
    print(f"mpmath版本：{mp.__version__ if hasattr(mp, '__version__') else '未知'}")
except ImportError as e:
    print(f"❌ 导入错误：{e}")
    print("请确保：")
    print("1. 在项目根目录运行")
    print("2. 已安装依赖：pip install mpmath")
    print("3. src/目录包含所有Python文件")

## 第一部分：计算核心常数

In [ ]:
# 计算谢尔宾斯基地毯的常数
print("计算谢尔宾斯基地毯常数...")

# 使用30位精度（足够用于分析）
constants = compute_sierpinski_constants(30)

D = constants['D']  # 维数 D = log₃8
S = constants['S']  # 面积 S = 8/9

print(f"✅ 常数计算完成")
print(f"维数 D = {mp.nstr(D, 20)}")
print(f"面积 S = {mp.nstr(S, 20)}")
print()
print(f"相关对数：")
print(f"ln2 = {mp.nstr(constants['ln2'], 20)}")
print(f"ln3 = {mp.nstr(constants['ln3'], 20)}")
print(f"ln8 = {mp.nstr(constants['ln8'], 20)}")

## 第二部分：连分数分析

In [ ]:
# 分析维数D的连分数
print("分析维数D的连分数指纹...")
print("-" * 50)

# 计算前100项连分数
cf_result = analyze_dimension_cf(dps=30, max_terms=100)

cf_D = cf_result['cf']
large_terms = cf_result['large_terms']

print(f"\n连分数长度：{len(cf_D)} 项")
print(f"前20项：{cf_D[:20]}")

if large_terms:
    print(f"\n🎯 发现巨大偏商（>100）：")
    for idx, val in large_terms:
        print(f"  第{idx}项: {val}")
else:
    print(f"\n❌ 未发现巨大偏商（>100）")
    
    # 查看有哪些较大的值
    large_values = [(i, v) for i, v in enumerate(cf_D) if v > 10]
    if large_values:
        print(f"较大的值（>10）：")
        for idx, val in large_values:
            print(f"  第{idx}项: {val}")

## 第三部分：验证代数关系

In [ ]:
# 验证理论关系：D = ln8/ln3
print("验证代数关系...")
print("-" * 50)

ln2 = constants['ln2']
ln3 = constants['ln3']
ln8 = constants['ln8']

# 关系1: D = ln8/ln3
relation1 = D * ln3 - ln8
print(f"D * ln3 - ln8 = {mp.nstr(relation1, 20)}")
print(f"理论值应为：0")
print(f"误差：{mp.nstr(abs(relation1), 10)}")

# 关系2: ln8 = 3*ln2
relation2 = 3 * ln2 - ln8
print(f"\n3*ln2 - ln8 = {mp.nstr(relation2, 20)}")
print(f"理论值应为：0")
print(f"误差：{mp.nstr(abs(relation2), 10)}")

## 第四部分：结果总结

In [ ]:
# 生成分析总结
print("分析总结")
print("=" * 50)

print(f"研究对象：谢尔宾斯基地毯")
print(f"1. 维数 D = log₃8 = {mp.nstr(D, 15)}")
print(f"2. 面积 S = 8/9 = {mp.nstr(S, 15)}")
print()

print(f"编译指纹分析：")
print(f"1. 连分数长度：{len(cf_D)} 项")
print(f"2. 前5项：{cf_D[:5]}")

if large_terms:
    print(f"3. 发现巨大偏商（>100）：")
    for idx, val in large_terms:
        print(f"   第{idx}项: {val}")
    
    # 计算巨大偏商的间隔
    if len(large_terms) >= 2:
        indices = [idx for idx, _ in large_terms]
        intervals = [indices[i+1] - indices[i] for i in range(len(indices)-1)]
        print(f"4. 巨大偏商间隔：{intervals}")
else:
    print(f"3. 未发现巨大偏商（>100）")
    
    # 显示最大值
    max_val = max(cf_D)
    max_idx = cf_D.index(max_val)
    print(f"4. 最大偏商：第{max_idx}项的{max_val}")

print()
print(f"代数关系验证：")
print(f"1. D = ln8/ln3 验证通过（误差：{mp.nstr(abs(relation1), 10)}）")
print(f"2. ln8 = 3*ln2 验证通过（误差：{mp.nstr(abs(relation2), 10)}）")

print("\n✅ 主分析完成！")

## 第五部分：保存结果

In [ ]:
# 保存分析结果
import json
from datetime import datetime

# 创建results目录
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

# 准备结果数据
result_data = {
    "analysis_time": datetime.now().isoformat(),
    "constants": {
        "D": mp.nstr(D, 30),
        "S": mp.nstr(S, 30),
        "ln2": mp.nstr(ln2, 30),
        "ln3": mp.nstr(ln3, 30),
        "ln8": mp.nstr(ln8, 30)
    },
    "continued_fraction": {
        "length": len(cf_D),
        "first_20_terms": cf_D[:20],
        "large_partials": large_terms if large_terms else []
    },
    "algebraic_relations": {
        "D_eq_ln8_over_ln3": {
            "value": mp.nstr(relation1, 20),
            "error": mp.nstr(abs(relation1), 20)
        },
        "ln8_eq_3_times_ln2": {
            "value": mp.nstr(relation2, 20),
            "error": mp.nstr(abs(relation2), 20)
        }
    }
}

# 保存为JSON文件
json_file = os.path.join(results_dir, "primary_analysis.json")
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(result_data, f, indent=2, ensure_ascii=False)

# 保存为文本文件（便于阅读）
txt_file = os.path.join(results_dir, "primary_analysis.txt")
with open(txt_file, 'w', encoding='utf-8') as f:
    f.write("谢尔宾斯基地毯主分析结果\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"分析时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("1. 核心常数：\n")
    f.write(f"   维数 D = {mp.nstr(D, 30)}\n")
    f.write(f"   面积 S = {mp.nstr(S, 30)}\n\n")
    
    f.write("2. 连分数分析：\n")
    f.write(f"   总项数：{len(cf_D)}\n")
    f.write(f"   前20项：{cf_D[:20]}\n")
    
    if large_terms:
        f.write(f"   巨大偏商（>100）：\n")
        for idx, val in large_terms:
            f.write(f"     第{idx}项: {val}\n")
    else:
        f.write(f"   未发现巨大偏商（>100）\n")
    f.write("\n")
    
    f.write("3. 代数关系验证：\n")
    f.write(f"   D = ln8/ln3 误差：{mp.nstr(abs(relation1), 20)}\n")
    f.write(f"   ln8 = 3*ln2 误差：{mp.nstr(abs(relation2), 20)}\n")

print(f"✅ 分析结果已保存：")
print(f"   JSON格式：{json_file}")
print(f"   文本格式：{txt_file}")